# Session 01: Practice Quiz

**Notebook 2 of 2: Self-check** · about 30 minutes

These questions check that you can *use* the ideas from `01-concepts.ipynb`, not just recognise them. For each one:

1. Write down your answer **before** opening "Show answer", or before running the code cell.
2. Run the cell where there is one.
3. Compare, and make sure you understand the *reasoning* in the answer, not only the result.

This quiz is formative: it's for you, and it isn't graded. Run it on the course environment's kernel (`.venv` in the course repository), the same as the concepts notebook.

## Q1 · Predict the output

What does this cell print? Will it raise an error, and if so, when?

In [ ]:
def sla_breached(ticket):
    return ticket["age_days"] > SLA_TARGET_DAYS


print("ready")

<details>
<summary>Show answer</summary>

It prints `ready` with **no error**.

Python compiles the whole cell first. That only checks *syntax*, and the syntax is valid. `SLA_TARGET_DAYS` is looked up when the `return` line *runs*, and the function is never called. If you add `sla_breached({"age_days": 3})`, you get `NameError: name 'SLA_TARGET_DAYS' is not defined` at that moment.

**Why it matters:** a Java or C# compiler would reject this file. In Python, linters (Ruff, Session 04) and tests catch this class of bug.

</details>

## Q2 · Predict the output

This time the source has a missing closing bracket on its **second** line. Does the first `print` run?

In [ ]:
source = 'print("step 1")\nprint("step 2"\n'

try:
    exec(compile(source, "deploy_check.py", "exec"))
except SyntaxError as error:
    print("SyntaxError on line", error.lineno)

<details>
<summary>Show answer</summary>

`step 1` is **not** printed. You only see `SyntaxError on line 2`.

Compilation covers the whole source before any of it executes, so a syntax error anywhere stops everything, including the perfectly valid line before it. Compare this with Q1: syntax errors are caught up front, while name errors wait until the line runs.

</details>

## Q3 · Predict the output

Running on the course kernel, what does this print, and what would it mean if the answer were the opposite?

In [ ]:
import sys

print(sys.prefix == sys.base_prefix)

<details>
<summary>Show answer</summary>

`False`. Inside a virtual environment, `sys.prefix` is the environment (`…/.venv`) and `sys.base_prefix` is the interpreter it was created from, so they differ.

If it printed `True`, the kernel would be running a base interpreter directly, with no virtual environment. Packages from the course's `uv.lock` would not be visible, and imports would start failing. Fix it with **Select Kernel → `.venv`** in VS Code, or by launching `uv run jupyter lab` from the repository root.

</details>

## Q4 · Engineering judgment: what goes in git?

Your `ops-toolkit` project folder contains:

`pyproject.toml` · `uv.lock` · `.python-version` · `main.py` · `README.md` · `.venv/` · `__pycache__/` · `.env`

Which do you commit, and why?

<details>
<summary>Show answer</summary>

| Commit | Don't commit |
|---|---|
| `pyproject.toml`: the declared dependencies | `.venv/`: machine-specific, large, rebuilt with `uv sync` |
| `uv.lock`: the exact versions the team shares | `__pycache__/`: interpreter-specific build output |
| `.python-version`: which interpreter to use | `.env`: local secrets (Session 05) |
| `main.py`, `README.md`: source and documentation | |

A useful rule: **commit inputs, ignore outputs.** `.venv/` and `__pycache__/` are *outputs* of `uv sync` and of the interpreter. `uv.lock` is also generated, but it's an *input* to every other machine's `uv sync`, which is why it's committed (like `package-lock.json`).

</details>

## Q5 · Debugging

A teammate pastes this from their terminal:

```text
PS C:\work\ops-toolkit> uv add httpx
Resolved 12 packages in 210ms
Installed 6 packages in 48ms
 + httpx==0.28.1
 ...
PS C:\work\ops-toolkit> python fetch_status.py
Traceback (most recent call last):
  File "C:\work\ops-toolkit\fetch_status.py", line 1, in <module>
    import httpx
ModuleNotFoundError: No module named 'httpx'
```

1. What is your hypothesis?
2. What single command would confirm it?
3. Give two different fixes.

<details>
<summary>Show answer</summary>

1. **Hypothesis:** `uv add` installed `httpx` into the project's `.venv`, but `python` on their `PATH` is a *different* (global) interpreter, which has its own `site-packages` without `httpx`. The traceback's last line names the failure, but not which interpreter was searching. That part you have to check.
2. **Confirm:** `python -c "import sys; print(sys.executable)"`. If the path isn't inside `C:\work\ops-toolkit\.venv`, the hypothesis holds. Compare it with `uv run python -c "import sys; print(sys.executable)"`.
3. **Fixes:**
   - Run it the uv way: `uv run fetch_status.py`. This is the recommended habit.
   - Or activate the environment first (`.venv\Scripts\Activate.ps1` on Windows, `source .venv/bin/activate` on macOS/Linux), then run `python fetch_status.py`.

   Running `pip install httpx` is **not** a fix. It would install into the global interpreter and recreate the shared-environment problem the project was set up to avoid.

</details>

## Q6 · Comparison

Map each concept to its Python/uv equivalent:

| You know | Python / uv equivalent |
|---|---|
| `package.json` / `pom.xml` / `.csproj` | ? |
| `package-lock.json` | ? |
| `node_modules/` | ? |
| `nvm install 22` / SDKMAN | ? |
| `npm ci` | ? |
| `npx some-tool` / `npm run` | ? |

<details>
<summary>Show answer</summary>

| You know | Python / uv equivalent |
|---|---|
| `package.json` / `pom.xml` / `.csproj` | `pyproject.toml` |
| `package-lock.json` | `uv.lock` |
| `node_modules/` | `.venv/` |
| `nvm install 22` / SDKMAN | `uv python install 3.13` (+ `.python-version`) |
| `npm ci` | `uv sync --locked` |
| `npx some-tool` / `npm run` | `uv run some-tool` |

One important difference has no equivalent row: in Node, per-project installs are the **default**. In Python, without uv or a virtual environment, installs are **global**.

</details>

## Q7 · Code completion

Complete `describe_environment()` so that it returns a dictionary with:

- `"python_version"`: a string such as `"3.13.1"`, built from `sys.version_info`
- `"executable"`: the path of the running interpreter
- `"in_virtual_environment"`: `True` or `False`

Then run the check cell below it.

In [ ]:
import sys


def describe_environment() -> dict:
    # Replace the line below with your implementation.
    return None

In [ ]:
result = describe_environment()

if result is None:
    print("Not implemented yet: complete describe_environment() above, then re-run both cells.")
else:
    assert set(result) == {"python_version", "executable", "in_virtual_environment"}, result.keys()
    assert result["python_version"].count(".") == 2, result["python_version"]
    assert result["executable"] == sys.executable
    assert result["in_virtual_environment"] is True, "Is this notebook running on the course .venv kernel?"
    print("All checks passed:", result)

<details>
<summary>Show answer</summary>

```python
def describe_environment() -> dict:
    major, minor, micro = sys.version_info[:3]
    return {
        "python_version": f"{major}.{minor}.{micro}",
        "executable": sys.executable,
        "in_virtual_environment": sys.prefix != sys.base_prefix,
    }
```

Notes on the choices:

- `sys.version_info[:3]` slices the first three fields, and *tuple unpacking* assigns them to three names in one line. You'll see both techniques throughout the course (Session 02).
- `sys.version.split()[0]` also works, but it parses a human-readable string. Building the value from the structured `version_info` is more robust.
- `sys.prefix != sys.base_prefix` is the standard virtual-environment check. You'll use it again in the workbook.

</details>

## Q8 · `__name__`

`audit.py` contains:

```python
print("loading audit")

def main():
    print("running audit")

if __name__ == "__main__":
    main()
```

What is printed by (a) `uv run audit.py`, and (b) `uv run python -c "import audit"` run in the same folder?

<details>
<summary>Show answer</summary>

(a) `loading audit`, then `running audit`. When a file is run as the program, its `__name__` is `"__main__"`, so the guarded block runs.

(b) Only `loading audit`. On import, `__name__` is `"audit"`, so `main()` is not called. The top-level `print` still runs, because importing a module **executes its top-level code** (once per process).

This is why real code keeps side effects out of module top level and puts the entry point behind the `__main__` guard. Tests (Session 04) import your modules, and importing must not start the program.

</details>

## Q9 · Engineering judgment: what version will they get?

`pyproject.toml` says `rich>=15.0.0`, and `uv.lock` pins `rich` at `15.0.0`. Six months later `rich 15.4.0` is released, and a new teammate clones the repository and runs `uv sync`.

1. Which version do they get?
2. How would you deliberately move the project to the newer release?

<details>
<summary>Show answer</summary>

1. **15.0.0.** `uv sync` installs what the **lock** says. The range in `pyproject.toml` only matters when the lock is re-resolved.
2. Run `uv lock --upgrade-package rich`, which re-resolves `rich` (and only what it needs) to the newest version allowed by the range. Run the application and tests, then commit the `uv.lock` change, ideally on its own so it is easy to review or revert.

This is the point of a lockfile: upgrades become a **decision with a diff**, not something that happens silently on someone's machine.

</details>

## Q10 · Engineering judgment: runtime or dev?

You are building an internal service that runs in production. Classify each dependency as **runtime** (`uv add`) or **development** (`uv add --dev`):

`httpx` (the service calls other APIs) · `ruff` · `pytest` · `rich` (used to format the service's CLI output) · `jupyterlab` (you explore data in notebooks)

<details>
<summary>Show answer</summary>

| Runtime: `uv add` | Development: `uv add --dev` |
|---|---|
| `httpx`: the code imports it in production | `ruff`: only linters and formatters use it |
| `rich`: the shipped CLI imports it | `pytest`: tests don't run in production |
| | `jupyterlab`: exploration, not part of the service |

The test is simple: **does production code import it?** Getting this right means a production install (`uv sync --locked --no-dev`) is smaller, faster and has less attack surface.

</details>

## Q11 · Debugging: a red CI build

CI runs `uv sync --locked` and fails:

```text
error: The lockfile at `uv.lock` needs to be updated, but `--locked` was provided.

hint: To update the lockfile, run `uv lock`.
```

The pull request's only dependency change is a new line, `"httpx>=0.28",`, typed directly into `pyproject.toml`. What happened, and what is the correct fix?

<details>
<summary>Show answer</summary>

`pyproject.toml` now asks for `httpx`, but `uv.lock` was never regenerated, so the two disagree. `--locked` makes uv **refuse** to silently re-resolve. That is exactly what you want on CI.

**Fix:** locally, run `uv lock` (or remove the hand-written line and use `uv add httpx`, which edits, locks and syncs in one step), run the code, and commit **both** `pyproject.toml` and `uv.lock`.

**Wrong fixes:** removing `--locked` from CI, which would hide future drift too, or editing `uv.lock` by hand.

</details>

## Q12 · Debugging: wrong model ID

A teammate wants to try Claude through OpenRouter. Their `.env` contains:

```text
LLM_PROVIDER=openrouter
LLM_MODEL=claude-haiku-5-5
OPENROUTER_API_KEY=<their key>
```

The client is created without complaint, but the first request fails with an error saying the model doesn't exist. The key is valid. What's wrong, and what are the two possible fixes?

<details>
<summary>Show answer</summary>

`claude-haiku-5-5` is **Anthropic's** ID for the model. OpenRouter names models `vendor/model`, and calls this one `anthropic/claude-haiku-5.5` (note the dot). The client was created fine because creating a client never contacts the provider. The mismatch only surfaced on the first real request.

Two fixes, both configuration-only:

1. Keep OpenRouter and use its ID: `LLM_MODEL=anthropic/claude-haiku-5.5`.
2. Call Anthropic directly: `LLM_PROVIDER=anthropic`, `LLM_MODEL=claude-haiku-5-5`, plus an `ANTHROPIC_API_KEY`.

Either way, restart the kernel afterwards. `load_dotenv()` doesn't overwrite variables that are already set.

</details>

## Q13 · Engineering judgment: a key in the code

A teammate's notebook contains:

```python
client = OpenAI(api_key="sk-proj-...")  # real key pasted here
```

They say: *"It's fine, the repository is private."* Give three reasons it isn't fine, and describe the correct setup.

<details>
<summary>Show answer</summary>

1. **Git history is forever.** Once committed, the key stays in every clone, fork and backup, even if a later commit deletes the line. "Private" repositories get shared, mirrored and made public by accident.
2. **Notebooks leak in more ways than code.** Outputs, screenshots, exported HTML and shared links all carry whatever the notebook contains.
3. **It couples code to one person's account and one provider.** Every teammate would need to edit the code to use their own key or a different provider.

**Correct setup:** keep the key in `.env` (git-ignored) or a real environment variable, and read it at run time, as the Part 8 cells do. Commit `.env.example` with empty placeholders so others know what to set. And because this key has already been pasted into a file, **revoke it in the provider's dashboard and create a new one**. Deleting the line doesn't un-leak it.

</details>

## Q14 · Debugging: mixed embeddings

Last week you embedded 5,000 runbook chunks with fastembed (`BAAI/bge-small-en-v1.5`) and saved the vectors. Today you set `EMBEDDING_PROVIDER=openai` in `.env`, and your search code fails:

```text
ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 1536 is different from 384)
```

1. What happened?
2. Why would it be *worse* if both models happened to produce vectors of the same length?
3. What's the fix, and how would you prevent this in future?

<details>
<summary>Show answer</summary>

1. The stored document vectors have 384 numbers each (fastembed's `bge-small-en-v1.5`). The new query vector has 1,536 (OpenAI's `text-embedding-3-small`). A dot product needs equal lengths, so NumPy refuses. The traceback's last line says exactly that: "size 1536 is different from 384".
2. With equal lengths, nothing would crash. You'd get similarity scores that look plausible and are meaningless, because each model arranges meaning in its own space. **A silent wrong answer is worse than a crash.**
3. **Fix:** re-embed all 5,000 chunks with the new model, or switch the query back to the model the documents were made with. **Prevention:** store the embedding model ID alongside the vectors, and check it on startup, refusing to search, or re-embedding automatically, when it differs from `EMBEDDING_MODEL`.

</details>

## How did you do?

- **Q1–Q3, Q8** cover the execution model. If any surprised you, re-read Parts 1 and 5 of the concepts notebook.
- **Q4, Q6, Q9–Q11** cover uv project management. These come up in every project for the rest of the course.
- **Q5** is the debugging pattern you will use most often. If you could not form a hypothesis, re-read Part 6.
- **Q12–Q13** cover LLM provider configuration and secrets. Revisit Part 8 before Session 09.
- **Q14** covers embeddings. Revisit Part 9 before Session 11.

Next: the workbook assignment in [`../workbook/assignment.md`](../workbook/assignment.md).